In [2]:
%load_ext autoreload
%autoreload 2
import d2l_local as d2l
import numpy as np
import torch
from torch.utils import data

In [3]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000) # 

输入：features（特征）和 labels（标签）—— 都是张量。

第一步：TensorDataset(*data_arrays) 把它们打包成一个数据集对象。这个对象就像一个“通讯录”：第 0 条记录对应 (features[0], labels[0])，第 1 条对应 (features[1], labels[1])……它按索引把特征和标签一一配对。

第二步：DataLoader(..., batch_size=10, shuffle=True) 在这个“通讯录”外面包装了一层，变成了一个 “批次取货机”。

DataLoader 是 PyTorch 专门提供的类，它内部实现了 Python 的 迭代器协议（即包含 __iter__ 方法）。所以它才能放进 for 循环里。

当 batch_size=10 时，它从 TensorDataset 里提取 10 个索引（比如 [3, 7, 1, ...]）。

根据索引去取配对好的数据，把 10 个特征拼成一个形状为 (10, 特征数) 的大张量 X，把 10 个标签拼成 (10,) 的大张量 y。

最后 return (X, y)。

In [4]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)
next(iter(data_iter))

[tensor([[ 0.6278, -0.3697],
         [ 0.0292,  0.1935],
         [-0.2828, -1.4418],
         [ 1.8903, -0.0039],
         [-2.2120, -0.4703],
         [ 0.3697,  0.2220],
         [ 0.0265, -1.7599],
         [ 0.5156, -0.2358],
         [ 0.4211, -0.3278],
         [-0.7257, -1.1337]]),
 tensor([[ 6.7245],
         [ 3.6083],
         [ 8.5359],
         [ 7.9846],
         [ 1.3753],
         [ 4.1699],
         [10.2371],
         [ 6.0465],
         [ 6.1397],
         [ 6.6110]])]

In [5]:
# nn是神经网络的缩写
from torch import nn
net = nn.Sequential(nn.Linear(2, 1))
#每个权重参数应该从均值为0、标准差为0.01的正态分布中随机采样， 偏置参数将初始化为零。
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

In [6]:
# MSELoss类，也称为平方范数。 默认情况下，它返回所有样本损失的平均值。
loss = nn.MSELoss()

小批量随机梯度下降算法是一种优化神经网络的标准工具， PyTorch在optim模块中实现了该算法的许多变种。 当我们实例化一个SGD实例时，我们要指定优化的参数 （可通过net.parameters()从我们的模型中获得）以及优化算法所需的超参数字典。 小批量随机梯度下降只需要设置lr值，这里设置为0.03。

In [7]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

In [8]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels) #打印一个epoch结束后的全局损失
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000199
epoch 2, loss 0.000101
epoch 3, loss 0.000101


In [9]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([0.0002, 0.0002])
b的估计误差： tensor([-7.5817e-05])
